In [ ]:
from __future__ import annotations
from pathlib import Path
import json, math, csv, gzip, sys, textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.container import BarContainer
from matplotlib.ticker import PercentFormatter, AutoLocator, MaxNLocator
from matplotlib.transforms import Bbox
from types import SimpleNamespace
ROOT=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'code').is_dir() and (p/'data').is_dir())
sys.path.insert(0,str(ROOT/'code/scripts'))
DATA=ROOT/'data/interim'
SUPP_DIR=ROOT/'deliverables/supplementary_S1-S9_20260907'
SUPP_DIR.mkdir(parents=True,exist_ok=True)
COL={'ink':'#17212B','muted':'#66727E','grid':'#D7DEE5',
     'blue':'#2F6B9A','blue_light':'#DCEAF4','green':'#14866D','green_light':'#D8EEE8',
     'orange':'#B26D16','orange_light':'#F4E7CF','red':'#D55E45','red_light':'#F7E1DB',
     'purple':'#6C5AA7','purple_light':'#E8E3F4','grey':'#7B8792','grey_light':'#E7EAED'}
COL.update(pdb=COL['blue'],pdb_light=COL['blue_light'],pos=COL['green'],pos_light=COL['green_light'],
           neg=COL['red'],neg_light=COL['red_light'],lit=COL['orange'],lit_light=COL['orange_light'],rand=COL['grey'])
mpl.rcParams.update({'font.family':'Arial','font.size':7,'font.weight':'normal',
    'axes.titleweight':'normal','axes.labelweight':'normal','axes.titlesize':7.5,
    'axes.labelsize':7,'xtick.labelsize':6,'ytick.labelsize':6,'legend.fontsize':5.8,
    'legend.frameon':False,'axes.linewidth':.6,'pdf.fonttype':42,'svg.fonttype':'none',
    'savefig.bbox':None,'figure.facecolor':'white','axes.facecolor':'white'})
def read_json(path): return json.loads(Path(path).read_text())
def ecdf(values):
    a=np.sort(np.asarray(values,float)); a=a[np.isfinite(a)]
    return a,np.arange(1,len(a)+1)/len(a)
def clean_axis(ax,grid=None):
    ax.spines['top'].set_visible(False);ax.spines['right'].set_visible(False)
    if grid: ax.grid(axis=grid,color=COL['grid'],lw=.45);ax.set_axisbelow(True)
clean=clean_axis
def panel_title(ax,title):
    ax.set_title(textwrap.fill(title,39),loc='left',pad=8,fontweight='normal')
def point_ci(ax,p,lo,hi,y,color,marker='o',size=22):
    ax.errorbar(p,y,xerr=[[max(0,p-lo)],[max(0,hi-p)]],fmt=marker,color=color,
                ecolor=COL['ink'],ms=np.sqrt(size),capsize=2,lw=.7)
def standardize_panel(ax,*_):
    # BarContainer records direction; never infer orientation from bar height.
    for container in ax.containers:
        if isinstance(container,BarContainer):
            for bar in container.patches:
                if container.orientation=='vertical':
                    width=bar.get_width();bar.set_x(bar.get_x()+width*.23);bar.set_width(width*.54)
                else:
                    height=bar.get_height();bar.set_y(bar.get_y()+height*.23);bar.set_height(height*.54)
                bar.set_linewidth(.5);bar.set_edgecolor(COL['ink'])
    ax.tick_params(width=.6,length=2.5)
def finish_panel(ax,label):
    standardize_panel(ax)
    ax.text(-.10,1.08,label,transform=ax.transAxes,fontsize=10,fontweight='normal',va='bottom')
    ax.set_xlabel(textwrap.fill(ax.get_xlabel(),42));ax.set_ylabel(textwrap.fill(ax.get_ylabel(),40))
    if isinstance(ax.xaxis.get_major_locator(),AutoLocator): ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    for artist in ax.findobj(mpl.text.Text):
        artist.set_fontfamily('Arial');artist.set_fontweight('normal')
    panel_axes[label]=ax
def bootstrap_mean_ci(values,seed=20260907,n_boot=10000):
    values=np.asarray(values,float); assert len(values)>0 and np.isfinite(values).all()
    rng=np.random.default_rng(seed); sims=[]
    for first in range(0,n_boot,250):
        ix=rng.integers(0,len(values),size=(min(250,n_boot-first),len(values)))
        sims.extend(values[ix].mean(axis=1))
    return tuple(np.quantile(sims,[.025,.975]))
def macro_result(df,ad,ac):
    d=df.dropna(subset=[ad,ac]).copy()
    d['w']=np.where(d[ad]>d[ac],1.,np.where(d[ad]<d[ac],0.,.5))
    vals=d.groupby('ac_id').w.mean().to_numpy()
    lo,hi=bootstrap_mean_ci(vals)
    return dict(p=float(vals.mean()),lo=float(lo),hi=float(hi),n=len(d),n_ac=len(vals))
def record(panel,items):
    pd.DataFrame(items).to_csv(SUPP_DIR/f'{FIGURE}_{panel}_source_data.tsv',sep='\t',index=False)

"""Render Supplementary Figures S1--S10 for the Figure 1--6 manuscript.

The script is deliberately downstream-only: it reuses frozen/interim tables and
published-model scores, performs no model training or inference, and writes a
machine-readable statistics ledger alongside publication-oriented PNG/PDF/SVG
files.  Supplementary analyses are limited to QC, resampling, stratification,
and sensitivity analyses that support the six main figures.
"""



import argparse
import csv
import gzip
import hashlib
import json
import math
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch, Polygon
from matplotlib.ticker import PercentFormatter

import analyze_figure5_score_drivers_v2 as f5
import analyze_negatome_figure4_subset_v1 as neg4


ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'code').is_dir() and (p/'data').is_dir())
OUT = ROOT / "results/figures/paper_supplement_v1"
DATA = ROOT / "data/interim"
SEEDS = [20260816, 20260817, 20260818]
ARM_COLORS = {"Original": COL["rand"], "Random-FT": COL["lit"], "Structure-FT": COL["neg"]}
SPLIT_LABELS = {
    "train_eligible_local_fit": "Local fit",
    "protein_unseen_family_seen": "Protein unseen,\ncluster seen",
    "family_unseen": "Cluster unseen",
}


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--output-dir", type=Path, default=OUT)
    p.add_argument("--project-root", type=Path, default=ROOT,
                   help="Project root containing data/interim (for copied per-figure renderers).")
    p.add_argument("--only", choices=("all", "S1", "S2", "S3", "S4", "S5", "S6", "S7", "S8", "S9", "S10"),
                   default="all")
    p.add_argument("--bootstrap", type=int, default=1000)
    p.add_argument("--seed", type=int, default=20260816)
    p.add_argument(
        "--swissprot", type=Path,
        default=Path("/data/chs/12.codebuddy/bridge/data/uniprot_sprot.fasta.gz"),
    )
    return p.parse_args()


def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


def read_table(path: Path) -> pd.DataFrame:
    return pd.read_csv(path, sep="\t" if path.suffix == ".tsv" else ",")


def save_all(fig: plt.Figure, outdir: Path, stem: str) -> None:
    for suffix in ("png", "pdf", "svg"):
        fig.savefig(outdir / f"{stem}.{suffix}", dpi=400)
    plt.close(fig)


def title(fig: plt.Figure, text: str, subtitle: str = "") -> None:
    fig.text(0.055, 0.978, text, fontsize=10.4, fontweight="normal", va="top")
    if subtitle:
        fig.text(0.055, 0.952, subtitle, fontsize=7.1, color=COL["ink"], va="top")


def annotate_bars(ax, bars, fmt="{:.0f}", dy=2.0, fontsize=6.1) -> None:
    for bar in bars:
        value = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, value + dy, fmt.format(value),
                ha="center", va="bottom", fontsize=fontsize)


def wilson(k: float, n: int, z: float = 1.96) -> tuple[float, float]:
    if n <= 0:
        return float("nan"), float("nan")
    p = k / n
    den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return centre - half, centre + half


def rank_summary(values: pd.DataFrame) -> dict:
    wins = np.where(values["ad"] > values["ac"], 1.0,
                    np.where(values["ad"] < values["ac"], 0.0, 0.5))
    margin = values["ad"].to_numpy() - values["ac"].to_numpy()
    return {
        "n": int(len(values)), "p": float(np.mean(wins)),
        "median_margin": float(np.median(margin)), "mean_margin": float(np.mean(margin)),
    }


def score_triplets(score_path: Path, eval_pairs: pd.DataFrame) -> pd.DataFrame:
    scores = read_table(score_path)
    merged = eval_pairs.merge(scores, on="directed_pair_id", how="inner", validate="one_to_one")
    wide = merged.pivot(index=["triplet_id", "split"], columns="role", values="score").reset_index()
    if not {"ac", "ad"}.issubset(wide.columns):
        raise RuntimeError(f"missing AC/AD roles in {score_path}")
    wide["win"] = np.where(wide["ad"] > wide["ac"], 1.0,
                           np.where(wide["ad"] < wide["ac"], 0.0, 0.5))
    wide["margin"] = wide["ad"] - wide["ac"]
    return wide


def load_family_score_tables() -> tuple[pd.DataFrame, dict[str, pd.DataFrame]]:
    base = DATA / "structure_family_transfer_v4/id50"
    pairs = read_table(base / "eval_pairs_v4.csv")
    tables = {"Original": score_triplets(base / "scores/original.csv", pairs)}
    for arm, prefix in (("Random-FT", "random"), ("Structure-FT", "structure")):
        for seed in SEEDS:
            tables[f"{arm}|{seed}"] = score_triplets(base / f"scores/{prefix}_seed{seed}.csv", pairs)
    return pairs, tables


def cluster_bootstrap_mean(values: np.ndarray, clusters: np.ndarray, n_boot: int,
                           rng: np.random.Generator) -> np.ndarray:
    unique = np.unique(clusters)
    grouped = {c: values[clusters == c] for c in unique}
    out = np.empty(n_boot)
    for i in range(n_boot):
        picked = rng.choice(unique, size=len(unique), replace=True)
        sample = np.concatenate([grouped[c] for c in picked])
        out[i] = float(sample.mean())
    return out


def read_structure_pair_scores() -> pd.DataFrame:
    pairs = read_table(DATA / "structure_triplet_scoring_v1/scoring_pairs_v1.tsv")
    scores = read_table(DATA / "structure_triplet_scoring_v1/plminteract_scores_v1.csv")
    if len(pairs) != len(scores):
        raise RuntimeError("structure pair/score row mismatch")
    pairs = pairs.copy(); pairs["score"] = scores.score.to_numpy()
    return pairs

def family_concentration(assign: pd.DataFrame, split: str) -> float:
    sub=assign[assign.split==split]
    if len(sub)==0: return float("nan")
    counts=Counter(pd.concat([sub.fam_a,sub.fam_c,sub.fam_d]).astype(str))
    return max(counts.values())/(3*len(sub))

def per_seed_records(tables: dict[str,pd.DataFrame], assignments: pd.DataFrame) -> pd.DataFrame:
    records=[]
    fam=assignments.set_index("triplet_id")["fam_a"].to_dict()
    for key,df in tables.items():
        arm,seed=(key.split("|",1)+["published"])[:2] if "|" in key else (key,"published")
        for split in SPLIT_LABELS:
            d=df[df.split==split]
            if len(d)==0: continue
            records.append({"arm":arm,"seed":str(seed),"split":split,"p":float(d.win.mean()),
                            "median_margin":float(d.margin.median()),"mean_margin":float(d.margin.mean()),"n":len(d),
                            "clusters":len({fam.get(t,t) for t in d.triplet_id})})
    return pd.DataFrame(records)

def load_negatome_rows(swissprot: Path) -> tuple[list[dict], list[dict], set[bytes]]:
    base=DATA/"negatome_manual_plminteract_v1"
    pairs=list(csv.DictReader((base/"negatome_manual_pairs_v1.tsv").open(encoding="utf-8"),delimiter="\t"))
    scores=[float(x["score"]) for x in csv.DictReader((base/"negatome_manual_scores_v1.csv").open())]
    accessions={x["protein_a"] for x in pairs}|{x["protein_b"] for x in pairs}
    tax=neg4.load_swissprot_tax(swissprot,accessions); rows=neg4.attach_flags(pairs,scores,tax,1024)
    seqrows=list(csv.DictReader((base/"negatome_manual_pairs_v1.csv").open()))
    trainkeys=neg4.load_string_positive_keys(ROOT/"data/training_reference/string_positive_sequence_pairs.sha1.tsv")
    for row,seq in zip(rows,seqrows): row["in_train_positive"]=neg4.seq_pair_key(seq["query"],seq["text"]) in trainkeys
    return rows,seqrows,trainkeys

def build_f5_matrix(rows: pd.DataFrame, fam_values: np.ndarray | None = None) -> tuple[np.ndarray,np.ndarray,list[str],dict]:
    raw={"log complex size":np.log(rows.num_protein_chains),"log minimum distance":np.log(rows.min_heavy),
         "training familiarity":(rows.train_familiarity/100),"A–C 3-mer similarity":rows.kmer_jaccard,
         "log pair length":np.log(rows.pair_tokens),"log1p replicate PDB":np.log1p(rows.n_pdb),
         "log1p bridge contacts":np.log1p(rows.bridge_contact_strength),"human":rows.human.astype(float)}
    cols=[np.ones(len(rows))]; names=["intercept"]; scaling={}
    for name,arr in raw.items():
        arr=np.asarray(arr,float)
        if name=="human": cols.append(arr); names.append(name); continue
        mean,sd=arr.mean(),arr.std(); cols.append((arr-mean)/sd); names.append(name); scaling[name]=(float(mean),float(sd))
    x=np.column_stack(cols); y=(rows.score>=.5).astype(float).to_numpy(); return x,y,names,scaling

def fit_logit_table(rows: pd.DataFrame, x: np.ndarray, y: np.ndarray, names: list[str], cluster: bool=True) -> list[dict]:
    beta,p=f5.logistic_irls(x,y); w=np.clip(p*(1-p),1e-7,None); bread=np.linalg.pinv((x.T*w)@x)
    cov=f5.two_way_cluster_cov(x,y-p,bread,rows.family_a.astype(str).tolist(),rows.family_c.astype(str).tolist()) if cluster else bread
    se=np.sqrt(np.clip(np.diag(cov),0,None)); return [{"term":n,"coef":float(b),"lo":float(b-1.96*s),"hi":float(b+1.96*s),"se":float(s)} for n,b,s in zip(names,beta,se)]

def fit_with_familiarity(rows: pd.DataFrame, fam: np.ndarray) -> dict:
    temp=rows.copy(); temp["train_familiarity"]=np.asarray(fam)*100
    x,y,names,_=build_f5_matrix(temp); table=fit_logit_table(temp,x,y,names,True)
    return next(r for r in table if r["term"]=="training familiarity")

def vif_values(x: np.ndarray,names:list[str]) -> list[dict]:
    out=[]
    for j in range(1,x.shape[1]):
        y=x[:,j]; other=np.delete(x,j,axis=1); beta=np.linalg.lstsq(other,y,rcond=None)[0]; pred=other@beta
        r2=1-np.sum((y-pred)**2)/np.sum((y-y.mean())**2); out.append({"term":names[j],"vif":float(1/max(1-r2,1e-12))})
    return out

def cluster_boot_rate(df:pd.DataFrame,group_col:str,n_boot:int,rng:np.random.Generator) -> tuple[float,float]:
    groups=df[group_col].astype(str).unique(); vals=[]
    grouped={g:df.loc[df[group_col].astype(str)==g,"positive_wins"].to_numpy(float) for g in groups}
    for _ in range(n_boot):
        picked=rng.choice(groups,size=len(groups),replace=True); sample=np.concatenate([grouped[g] for g in picked]); vals.append(sample.mean())
    return tuple(np.percentile(vals,[2.5,97.5]))



from pathlib import Path
import json, math, csv, gzip, sys, textwrap
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Rectangle
from matplotlib.container import BarContainer
from matplotlib.ticker import PercentFormatter, AutoLocator, MaxNLocator
from matplotlib.transforms import Bbox
from types import SimpleNamespace
ROOT=next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'code').is_dir() and (p/'data').is_dir())
sys.path.insert(0,str(ROOT/'code/scripts'))
DATA=ROOT/'data/interim'
SUPP_DIR=ROOT/'deliverables/supplementary_S1-S9_20260907'
SUPP_DIR.mkdir(parents=True,exist_ok=True)
COL={'ink':'#17212B','muted':'#66727E','grid':'#D7DEE5',
     'blue':'#2F6B9A','blue_light':'#DCEAF4','green':'#14866D','green_light':'#D8EEE8',
     'orange':'#B26D16','orange_light':'#F4E7CF','red':'#D55E45','red_light':'#F7E1DB',
     'purple':'#6C5AA7','purple_light':'#E8E3F4','grey':'#7B8792','grey_light':'#E7EAED'}
COL.update(pdb=COL['blue'],pdb_light=COL['blue_light'],pos=COL['green'],pos_light=COL['green_light'],
           neg=COL['red'],neg_light=COL['red_light'],lit=COL['orange'],lit_light=COL['orange_light'],rand=COL['grey'])
mpl.rcParams.update({'font.family':'Arial','font.size':7,'font.weight':'normal',
    'axes.titleweight':'normal','axes.labelweight':'normal','axes.titlesize':7.5,
    'axes.labelsize':7,'xtick.labelsize':6,'ytick.labelsize':6,'legend.fontsize':5.8,
    'legend.frameon':False,'axes.linewidth':.6,'pdf.fonttype':42,'svg.fonttype':'none',
    'savefig.bbox':None,'figure.facecolor':'white','axes.facecolor':'white'})
def read_json(path): return json.loads(Path(path).read_text())
def ecdf(values):
    a=np.sort(np.asarray(values,float)); a=a[np.isfinite(a)]
    return a,np.arange(1,len(a)+1)/len(a)
def clean_axis(ax,grid=None):
    ax.spines['top'].set_visible(False);ax.spines['right'].set_visible(False)
    if grid: ax.grid(axis=grid,color=COL['grid'],lw=.45);ax.set_axisbelow(True)
clean=clean_axis
def panel_title(ax,title):
    ax.set_title(textwrap.fill(title,39),loc='left',pad=8,fontweight='normal')
def point_ci(ax,p,lo,hi,y,color,marker='o',size=22):
    ax.errorbar(p,y,xerr=[[max(0,p-lo)],[max(0,hi-p)]],fmt=marker,color=color,
                ecolor=COL['ink'],ms=np.sqrt(size),capsize=2,lw=.7)
def standardize_panel(ax,*_):
    # BarContainer records direction; never infer orientation from bar height.
    for container in ax.containers:
        if isinstance(container,BarContainer):
            for bar in container.patches:
                if container.orientation=='vertical':
                    width=bar.get_width();bar.set_x(bar.get_x()+width*.23);bar.set_width(width*.54)
                else:
                    height=bar.get_height();bar.set_y(bar.get_y()+height*.23);bar.set_height(height*.54)
                bar.set_linewidth(.5);bar.set_edgecolor(COL['ink'])
    ax.tick_params(width=.6,length=2.5)
def finish_panel(ax,label):
    standardize_panel(ax)
    ax.text(-.10,1.08,label,transform=ax.transAxes,fontsize=10,fontweight='normal',va='bottom')
    ax.set_xlabel(textwrap.fill(ax.get_xlabel(),42));ax.set_ylabel(textwrap.fill(ax.get_ylabel(),40))
    if isinstance(ax.xaxis.get_major_locator(),AutoLocator): ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    for artist in ax.findobj(mpl.text.Text):
        artist.set_fontfamily('Arial');artist.set_fontweight('normal')
    panel_axes[label]=ax
def bootstrap_mean_ci(values,seed=20260907,n_boot=10000):
    values=np.asarray(values,float); assert len(values)>0 and np.isfinite(values).all()
    rng=np.random.default_rng(seed); sims=[]
    for first in range(0,n_boot,250):
        ix=rng.integers(0,len(values),size=(min(250,n_boot-first),len(values)))
        sims.extend(values[ix].mean(axis=1))
    return tuple(np.quantile(sims,[.025,.975]))
def macro_result(df,ad,ac):
    d=df.dropna(subset=[ad,ac]).copy()
    d['w']=np.where(d[ad]>d[ac],1.,np.where(d[ad]<d[ac],0.,.5))
    vals=d.groupby('ac_id').w.mean().to_numpy()
    lo,hi=bootstrap_mean_ci(vals)
    return dict(p=float(vals.mean()),lo=float(lo),hi=float(hi),n=len(d),n_ac=len(vals))
def record(panel,items):
    pd.DataFrame(items).to_csv(SUPP_DIR/f'{FIGURE}_{panel}_source_data.tsv',sep='\t',index=False)


In [ ]:
FIGURE='S5'
LAYOUT={'width_mm':180,'height_mm':245,'left':.16,'right':.975,'bottom':.075,'top':.925,'wspace':.70,'hspace':.70}
fig=plt.figure(figsize=(LAYOUT['width_mm']/25.4,LAYOUT['height_mm']/25.4))
gs=fig.add_gridspec(3,2,left=LAYOUT['left'],right=LAYOUT['right'],bottom=LAYOUT['bottom'],top=LAYOUT['top'],wspace=LAYOUT['wspace'],hspace=LAYOUT['hspace'])
fig.text(.05,.985,'Supplementary Figure S5 | '+textwrap.fill('Structural-negative fine-tuning and transfer robustness',77),fontsize=8,va='top',fontweight='normal')
panel_axes={}


In [ ]:
from types import SimpleNamespace
args=SimpleNamespace(seed=20260816,bootstrap=1000)
stats={}
outdir=SUPP_DIR
ARM_COLORS={'Original':COL['grey'],'Random-FT':COL['blue'],'Structure-FT':COL['green']}
_,tables=load_family_score_tables()
base=DATA/"structure_family_transfer_v4/id50"; assign=read_table(base/"triplet_assignments_v4.tsv")
rec=per_seed_records(tables,assign)
histories=[]
for arm,prefix in [("Random-FT","random"),("Structure-FT","structure")]:
    for seed in SEEDS:
        h=read_json(base/f"ft_{prefix}_seed{seed}/training_summary_v1.json")
        for row in h["history"]: histories.append({"arm":arm,"seed":seed,**row})
hist=pd.DataFrame(histories)




In [ ]:
# S5a: source notebook 5, panel cell S5a
if 'a' in panel_axes: panel_axes['a'].remove()
ax=fig.add_subplot(gs[0,0])

panel_title(ax,'Per-seed family-unseen ranking')
subset=rec[rec.split.eq('family_unseen')]
for i,(arm,color) in enumerate(ARM_COLORS.items()):
    vals=subset[subset.arm.eq(arm)].p.to_numpy()
    ax.scatter(vals,np.full(len(vals),i)+np.linspace(-.08,.08,len(vals)),s=24,color=color,edgecolor='white',lw=.4)
ax.set_yticks(range(3));ax.set_yticklabels(list(ARM_COLORS));ax.invert_yaxis();ax.axvline(.5,color=COL['grey'],ls='--',lw=.7)
ax.set(xlim=(.45,.60),xlabel='P(D>C), n=540 family-unseen triplets');ax.xaxis.set_major_formatter(PercentFormatter(1));clean_axis(ax,'x')
record('a',subset.to_dict('records'))

finish_panel(ax,'a')


In [ ]:
# S5b: source notebook 5, panel cell S5b
if 'b' in panel_axes: panel_axes['b'].remove()
ax=fig.add_subplot(gs[0,1])

panel_title(ax,'Family-unseen median score margins')
subset=rec[rec.split.eq('family_unseen')]
for i,(arm,color) in enumerate(ARM_COLORS.items()):
    vals=subset[subset.arm.eq(arm)].median_margin.to_numpy()
    ax.scatter(vals,np.full(len(vals),i)+np.linspace(-.08,.08,len(vals)),s=24,color=color,edgecolor='white',lw=.4)
ax.set_yticks(range(3));ax.set_yticklabels(list(ARM_COLORS));ax.invert_yaxis();ax.axvline(0,color=COL['grey'],ls='--',lw=.7)
ax.set(xlim=(-.01,.01),xlabel='Median score(A,D) − score(A,C)');ax.ticklabel_format(axis='x',style='plain',useOffset=False);clean_axis(ax,'x')
record('b',subset.to_dict('records'))

finish_panel(ax,'b')


In [ ]:
# S5c: source notebook 5, panel cell 14
if 'c' in panel_axes: panel_axes['c'].remove()
ax=fig.add_subplot(gs[1,0])
panel_title(ax,"Training loss")
for (arm,seed),d in hist.groupby(["arm","seed"]): ax.plot(d.epoch,d.train_loss,color=ARM_COLORS[arm],alpha=.55,lw=1,marker="o",ms=3)
ax.set_xticks([1,2,3]); ax.set_xlabel("Epoch"); ax.set_ylabel("Training loss"); clean_axis(ax,"both")

finish_panel(ax,'c')


In [ ]:
stats={}
outdir=SUPP_DIR
ARM_COLORS={'Original':COL['grey'],'Random-FT':COL['blue'],'Structure-FT':COL['green']}
_,tables=load_family_score_tables()
assign=read_table(DATA/"structure_family_transfer_v4/id50/triplet_assignments_v4.tsv")
meta=assign.set_index("triplet_id")
records=[]

def averaged_score_table(arm: str) -> pd.DataFrame:
    if arm == "Original":
        return tables["Original"].copy()
    frames = [tables[f"{arm}|{seed}"][["triplet_id", "split", "ac", "ad"]] for seed in SEEDS]
    merged = pd.concat(frames, ignore_index=True).groupby(
        ["triplet_id", "split"], as_index=False
    )[["ac", "ad"]].mean()
    merged["win"] = np.where(merged.ad > merged.ac, 1.0,
                             np.where(merged.ad < merged.ac, 0.0, 0.5))
    merged["margin"] = merged.ad - merged.ac
    return merged

for arm in ["Original","Random-FT","Structure-FT"]:
    df=averaged_score_table(arm).join(meta[["replicated_noncontact","fam_a","fam_c","fam_d"]],on="triplet_id")
    dominant = (df[["fam_a", "fam_c", "fam_d"]] == "P40227").any(axis=1)
    for name,mask in [("All core",df.split=="family_unseen"),("Replicated ≥2 PDB",(df.split=="family_unseen")&(df.replicated_noncontact==1)),
                      ("Family seen",df.split=="protein_unseen_family_seen"),("Leave P40227 out",(df.split=="protein_unseen_family_seen")&(~dominant))]:
        d=df[mask]; records.append({"arm":arm,"run":"published" if arm=="Original" else "mean-of-three-seed-scores",
                                    "subset":name,"p":float(d.win.mean()),"median_margin":float(d.margin.median()),"n":len(d)})
rec=pd.DataFrame(records)




In [ ]:
# S5d: source notebook 6, panel cell S5d
if 'd' in panel_axes: panel_axes['d'].remove()
ax=fig.add_subplot(gs[1,1])

panel_title(ax,'Aggregate family-unseen fine-tuning effects')
base=averaged_score_table('Original');base=base[base.split.eq('family_unseen')].set_index('triplet_id')
rows=[]
for i,arm in enumerate(['Random-FT','Structure-FT']):
    other=averaged_score_table(arm).set_index('triplet_id').loc[base.index]
    delta=other.win.to_numpy()-base.win.to_numpy();families=meta.loc[base.index,'fam_a'].to_numpy()
    sims=cluster_bootstrap_mean(delta,families,10000,np.random.default_rng(20260907));lo,hi=np.quantile(sims,[.025,.975])
    point_ci(ax,delta.mean(),lo,hi,i,ARM_COLORS[arm]);rows.append(dict(arm=arm,n=len(delta),delta=float(delta.mean()),lo=float(lo),hi=float(hi)))
ax.set_yticks(range(2));ax.set_yticklabels(['Random-FT','Structure-FT']);ax.invert_yaxis();ax.axvline(0,color=COL['grey'],ls='--',lw=.7)
ax.set(xlim=(-.08,.08),xlabel='ΔP(D>C) versus original model');ax.xaxis.set_major_formatter(PercentFormatter(1));clean_axis(ax,'x');record('d',rows)

finish_panel(ax,'d')


In [ ]:
# S5e: source notebook 6, panel cell S5e
if 'e' in panel_axes: panel_axes['e'].remove()
ax=fig.add_subplot(gs[2,0])

panel_title(ax,'Repeated-PDB family-unseen sensitivity')
rows=[]
for i,(arm,color) in enumerate(ARM_COLORS.items()):
    df=averaged_score_table(arm).join(meta[['replicated_noncontact','fam_a']],on='triplet_id')
    for off,subset in [(-.12,'All core'),(.12,'≥2 PDB')]:
        d=df[df.split.eq('family_unseen')]
        if subset=='≥2 PDB': d=d[d.replicated_noncontact.eq(1)]
        boots=cluster_bootstrap_mean(d.win.to_numpy(),d.fam_a.to_numpy(),10000,np.random.default_rng(20260907));lo,hi=np.quantile(boots,[.025,.975])
        point_ci(ax,d.win.mean(),lo,hi,i+off,color,marker='o' if subset=='All core' else 's');rows.append(dict(arm=arm,subset=subset,n=len(d),p=d.win.mean(),lo=lo,hi=hi))
ax.set_yticks(range(3));ax.set_yticklabels(list(ARM_COLORS));ax.invert_yaxis();ax.axvline(.5,color=COL['grey'],ls='--',lw=.7)
ax.plot([],[],marker='o',ls='',color=COL['ink'],label='All core (n=540)');ax.plot([],[],marker='s',ls='',color=COL['ink'],label='≥2 PDB (n=353)');ax.legend(loc='lower right')
ax.set(xlim=(.40,.65),xlabel='P(D>C)');ax.xaxis.set_major_formatter(PercentFormatter(1));clean_axis(ax,'x');record('e',rows)

finish_panel(ax,'e')


In [ ]:
# S5f: source notebook 6, panel cell S5f
if 'f' in panel_axes: panel_axes['f'].remove()
ax=fig.add_subplot(gs[2,1])

panel_title(ax,'Dominant-family removal (family-seen)')
rows=[]
for i,(arm,color) in enumerate(ARM_COLORS.items()):
    df=averaged_score_table(arm).join(meta[['fam_a','fam_c','fam_d']],on='triplet_id')
    d0=df[df.split.eq('protein_unseen_family_seen')]
    for off,subset in [(-.12,'All family-seen'),(.12,'P40227 removed')]:
        d=d0 if subset=='All family-seen' else d0[~d0[['fam_a','fam_c','fam_d']].eq('P40227').any(axis=1)]
        boots=cluster_bootstrap_mean(d.win.to_numpy(),d.fam_a.to_numpy(),10000,np.random.default_rng(20260907));lo,hi=np.quantile(boots,[.025,.975])
        point_ci(ax,d.win.mean(),lo,hi,i+off,color,marker='o' if subset=='All family-seen' else 's');rows.append(dict(arm=arm,subset=subset,n=len(d),p=d.win.mean(),lo=lo,hi=hi))
ax.set_yticks(range(3));ax.set_yticklabels(list(ARM_COLORS));ax.invert_yaxis();ax.axvline(.5,color=COL['grey'],ls='--',lw=.7)
ax.plot([],[],marker='o',ls='',color=COL['ink'],label='All (n=101)');ax.plot([],[],marker='s',ls='',color=COL['ink'],label='P40227 removed (n=78)');ax.legend(loc='lower right')
ax.set(xlim=(.25,.85),xlabel='P(D>C)');ax.xaxis.set_major_formatter(PercentFormatter(1));clean_axis(ax,'x');record('f',rows)

finish_panel(ax,'f')


In [ ]:

for artist in fig.findobj(mpl.text.Text):artist.set_fontfamily('Arial');artist.set_fontweight('normal')
fig.canvas.draw()
for label,axis in panel_axes.items():
    bbox=axis.get_tightbbox(fig.canvas.get_renderer()).expanded(1.04,1.04).transformed(fig.dpi_scale_trans.inverted())
    for ext in ('pdf','svg','png'):
        fig.savefig(SUPP_DIR/f'{FIGURE}_{label}.{ext}',dpi=600,bbox_inches=bbox)
for ext in ('pdf','svg','png'):fig.savefig(SUPP_DIR/f'{FIGURE}.{ext}',dpi=600)
plt.close(fig)
